# Schülerzahlen Prognose-Modell

Dieses Notebook analysiert die historischen Schülerzahlen des BBZ Rendsburg-Eckernförde und erstellt eine Prognose bis 2030.

## 1. Bibliotheken & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet

# Styling
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis")
pd.options.display.float_format = '{:,.0f}'.format

## 2. Daten laden und vorbereiten

In [ ]:
# Schülerdaten laden
df = pd.read_csv('../data/processed/schuelerzahlen_aggregated.csv')

# Jahr zu Datum konvertieren (für Prophet notwendig: 'ds' Spalte)
df['ds'] = pd.to_datetime(df['jahr'].astype(str) + '-09-01')
df['y'] = df['schuelerzahl']

print("Schülerdaten geladen:")
display(df.tail())

In [ ]:
# --- Bevölkerungsdaten laden und mergen ---

def load_population_data(filepath):
    raw_data = pd.read_csv(filepath, sep=';', header=None, dtype=str)
    year_row_idx = raw_data[raw_data[0] == 'Jahr'].index[0]
    years = raw_data.iloc[year_row_idx].values
    sep_row_idx = raw_data[raw_data[0] == 'September'].index[0]
    values = raw_data.iloc[sep_row_idx].values
    pop_data = pd.DataFrame({'jahr_str': years, 'bevoelkerung': values})
    pop_data = pop_data[pop_data['jahr_str'].str.match(r'^\d{4}$', na=False)]
    pop_data['jahr'] = pop_data['jahr_str'].astype(int)
    pop_data['bevoelkerung'] = pd.to_numeric(pop_data['bevoelkerung'], errors='coerce')
    return pop_data[['jahr', 'bevoelkerung']].dropna()

file_2011 = '../data/raw/monatszahlen_bevoelkerung_am_monatsende_insgesamt_basis_zensus_2011_05-02-2026.csv'
df_pop = load_population_data(file_2011)
df = pd.merge(df, df_pop, on='jahr', how='left')
df['bevoelkerung'] = df['bevoelkerung'].interpolate()

print("Daten inkl. Bevölkerung:")
display(df.tail())

## 3. Explorative Analyse (Ist-Zustand)

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='jahr', y='y', marker='o', linewidth=2, color='#2c3e50')
plt.title('Entwicklung der Schülerzahlen (2009-2025)', fontsize=16)
plt.ylabel('Anzahl Schüler')
plt.xlabel('Schuljahr (Beginn)')
plt.grid(True, alpha=0.3)

# Annotationen für wichtige Ereignisse
plt.annotate('Peak (3.135)', xy=(2016, 3135), xytext=(2016, 3200),
             arrowprops=dict(facecolor='black', shrink=0.05), ha='center')

plt.annotate('Aktuell (2.594)', xy=(2025, 2594), xytext=(2025, 2700),
             arrowprops=dict(facecolor='red', shrink=0.05), ha='center', color='red')

plt.show()

## 4. Prognose mit Prophet

Wir erstellen eine Prognose für die nächsten 5 Jahre (bis 2030). Prophet berücksichtigt automatisch den historischen Trend.

**Annahmen:**
*   Der Trend der letzten Jahre setzt sich grundsätzlich fort.
*   Saisonale Schwankungen werden geglättet.

In [ ]:
# Modell initialisieren und trainieren
# WICHTIG: Bei jährlichen Datenpunkten KEINE Saisonalität modellieren!
m = Prophet(
    growth='linear',
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05  # Weniger sensitiv auf letzte Änderungen
)

# Fit
m.fit(df[['ds', 'y']])

# Zukunft dataframe erstellen (5 Jahre)
future = m.make_future_dataframe(periods=5, freq='YS') # YS = Year Start
forecast = m.predict(future)

# Plotten
fig1 = m.plot(forecast)
plt.title('Prophet Prognose bis 2030')
plt.xlabel('Jahr')
plt.ylabel('Schülerzahl')
plt.show()

## 5. Szenario-Analyse

Basierend auf der Analyse gibt es drei Haupttreiber für zukünftiges Wachstum:
1.  **Gesundheit & Soziales (+++)**: Starkes Wachstum durch Fachkräftemangel und PiA.
2.  **Technik (+)**: Stabilisierung/Wachstum durch Energiewende.
3.  **Wirtschaft (o)**: Stabile Entwicklung.

Wir simulieren hier ein **"Wachstums-Szenario"**, bei dem wir davon ausgehen, dass diese Maßnahmen den negativen Trend umkehren.

In [ ]:
# Berechnung eines optimistischen Szenarios
# Annahme: Ab 2026 jährliches Wachstum von ca. 1.5% (statt Trendfortschreibung)

last_real_value = df.iloc[-1]['y']
years_future = [2026, 2027, 2028, 2029, 2030]
growth_rate = 0.015 # 1.5% Wachstum pro Jahr

scenario_values = []
current_val = last_real_value
for year in years_future:
    current_val = current_val * (1 + growth_rate)
    scenario_values.append({'jahr': year, 'y_scenario': current_val})

df_scenario = pd.DataFrame(scenario_values)

# Visualisierung Vergleich
plt.figure(figsize=(12, 6))

# Historische Daten
plt.plot(df['jahr'], df['y'], label='Historisch', marker='o', color='black')

# Prophet Forecast (Trend)
forecast_filtered = forecast[forecast['ds'].dt.year > 2025]
plt.plot(forecast_filtered['ds'].dt.year, forecast_filtered['yhat'], 
         label='Prophet Trend (Status Quo)', linestyle='--', color='red')

# Optimistisches Szenario
plt.plot(df_scenario['jahr'], df_scenario['y_scenario'], 
         label='Szenario: Gezieltes Wachstum (+1.5% p.a.)', linestyle='--', color='green', marker='x')

plt.title('Szenarien-Vergleich bis 2030', fontsize=16)
plt.xlabel('Jahr')
plt.ylabel('Schülerzahl')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Fazit der Szenarien

*   **Rote Linie (Status Quo):** Wenn wir nichts ändern, folgt das Modell dem aktuellen Abwärtstrend (getrieben durch den starken Rückgang 2025).
*   **Grüne Linie (Wachstum):** Wenn die Maßnahmen in Gesundheit/Soziales und Technik greifen, können wir den Trend umkehren.

**Nächste Schritte:**
Detaillierung der Daten nach Fachbereichen, um spezifischere Prognosen zu erstellen.